In [1]:
from types import SimpleNamespace
import numpy as np, pandas as pd, scanpy as sc
from torch.utils.data import DataLoader

from codes.sprint.training import *
from codes.sprint.data import *
from codes.sprint.utils import *


In [2]:
args = SimpleNamespace(
    schemes=["A2"],
    epochs=30,
    batch_size=16,
    lr=3e-5,
    image_lr_factor=0.1,
    warmup_epochs=3,
    min_lr_factor=0.05,
    weight_decay=1e-4,
    mse_weight=0.0,
    pcc_weight=1.0,
    grad_clip=1.0,
    freeze_image=False,
    resume=True,
    tag="unfreeze_zscore_pcc30_warmup_cosine",
)


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGE_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
print(f"Device: {DEVICE}")

Device: cuda


In [4]:
train_path = "datas/msi/mouse_brain_1/mouse_brain_1_Processed.h5ad"
val_path = "datas/msi/mouse_brain_2/mouse_brain_2_Processed.h5ad"

In [6]:
ad_train = sc.read_h5ad(train_path)
names = [str(x) for x in list(ad_train.uns["metabolite_names"])]
y_train = ad_train.obsm["metabolite_expression_log"]
y_train = y_train.values if hasattr(y_train, "values") else y_train
y_train = np.asarray(y_train, dtype=np.float32)
target_mean = y_train.mean(axis=0).astype(np.float32)
target_std = np.maximum(y_train.std(axis=0), 1e-3).astype(np.float32)
num_genes = ad_train.n_vars
num_metabolites = len(names)
del ad_train, y_train

In [7]:
train_ds = MouseBrainMSIDataset(
    train_path,
    names,
    target_mean,
    target_std,
    image_norm=True,
)
val_ds = MouseBrainMSIDataset(
    val_path,
    names,
    target_mean,
    target_std,
    image_norm=True,
)
train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)

print(f"Genes: {num_genes}")
print(f"Metabolites: {num_metabolites}")
print(f"Train samples: {len(train_ds)}")
print(f"Val samples: {len(val_ds)}")


Genes: 32285
Metabolites: 1538
Train samples: 2658
Val samples: 3098


In [9]:
output_dirs = {
    "A2": Path("models_mouse_brain/improved_unfreeze_zscore_pcc30_warmup_cosine/A2"),
}
summary_path = Path("models_mouse_brain/improved_unfreeze_zscore_pcc30_warmup_cosine/summary_metrics.csv")

In [10]:
args.schemes

['A2']

In [ ]:
summaries = []
for scheme in args.schemes:
    out_dir = output_dirs[scheme]
    out_dir.mkdir(parents=True, exist_ok=True)
    _, metrics, history = train_one(
        args,
        scheme,
        train_loader,
        val_loader,
        num_genes,
        num_metabolites,
        target_mean,
        target_std,
        out_dir
    )
    save_per_metabolite(metrics["preds"], metrics["targets"], names, out_dir / "per_metabolite_metrics.csv")
    summaries.append(
        {
            "Model": scheme,
            "BestPCC": max(history["val_pcc"]),
            "FinalBestEvalPCC": metrics["pcc"],
            "RMSE": metrics["rmse"],
            "R2": metrics["r2"],
        }
    )

df = pd.DataFrame(summaries)
df.to_csv(summary_path, index=False)
print("Summary")
print(df.to_string(index=False))
df
